# Agents

An **agent** is a loop where a model: 

1. Reads user input + context

2. Decides whether to call _tools_ (code you provide)

3. Uses tool results to reason 

4. Replies or takes another action


OpenAI’s platform supports this with:

* Responses API + function tools (a.k.a. function calling) to let the model call your code,
* Embeddings for retrieval/memory, and
* An Agents SDK that wraps common agentic patterns (tools, state, tracing) to speed development.  

## Quick start: a minimal “chat agent”

Goal: one file, streams tokens, easy to run.

```python
# pip install openai==1.*  (or the official SDK version in your env)
import os, sys
from openai import OpenAI
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def chat(prompt):
    # Responses API: simple chat completion with streaming
    with client.responses.stream(
        model="gpt-4.1-mini",
        input=[{"role": "user", "content": prompt}],
    ) as stream:
        for event in stream:
            if event.type == "response.output_text.delta":
                sys.stdout.write(event.delta)
        print()  # newline

if __name__ == "__main__":
    while True:
        try:
            chat(input("> "))
        except (EOFError, KeyboardInterrupt):
            break

## 2) Give your agent tools (function calling)

Tools = your functions. The model chooses when/with what arguments to call them; you execute and return results back into the conversation.

The following example adds a calculator tool. 

```python
from typing import Dict, Any
import json, math
from openai import OpenAI
client = OpenAI()

def calc(expression: str) -> str:
    return str(eval(expression, {"__builtins__": {}}, {"pi": math.pi, "e": math.e}))

tools = [
    {
        "type": "function",
        "function": {
            "name": "calc",
            "description": "Evaluate a safe math expression. Supports pi and e.",
            "parameters": {
                "type": "object",
                "properties": {"expression": {"type": "string"}},
                "required": ["expression"],
                "additionalProperties": False,
            },
        },
    }
]

def agent_turn(user_text: str):
    msg = [{"role":"user","content":user_text}]
    # 1) Ask the model; allow tool use
    r = client.responses.create(
        model="gpt-4.1",
        input=msg,
        tools=tools
    )

    # 2) If the model wants to call a tool, execute & return result
    while True:
        out = r.output  # list of content blocks / tool calls / text
        tool_call = next((b for b in out if b.type=="tool_call"), None)
        if not tool_call:
            # no tool call -> print any text parts
            print("".join(b.text for b in out if b.type=="output_text"))
            break

        if tool_call.tool_name == "calc":
            args = json.loads(tool_call.arguments)
            result = calc(**args)
            # 3) Send tool result back as tool output & continue the loop
            r = client.responses.create(
                model="gpt-4.1",
                input=[
                    {"role":"user","content":user_text},
                    {"role":"tool", "tool_call_id": tool_call.id, "content": result}
                ],
                tools=tools
            )

agent_turn("What is sin(pi/2)^2 + cos(0)^2 ? Use the calculator.")
```

Key ideas:

•	You declare tool schema; the model emits a function call with arguments.

•	You run the function and send a tool response back for the model to finish the thought.

Details: function tools / arguments / streaming formats are in the API reference. 

## 3) Retrieval-Augmented Generation (RAG): give your agent knowledge

Use Embeddings to index your documents, then do a vector search each turn and stuff the best snippets into context.

```python

# pip install numpy tiktoken openai
import numpy as np, tiktoken
from openai import OpenAI
client = OpenAI()

docs = [
  {"id":"p1","text":"Furman University is in Greenville, SC."},
  {"id":"p2","text":"CSC-475 is an applied data science capstone with BMW collaboration."},
  # add your corpus (or load from disk)
]

# build embeddings once
E_MODEL = "text-embedding-3-small"
vectors = client.embeddings.create(
    model=E_MODEL,
    input=[d["text"] for d in docs]
).data
emb = np.vstack([np.array(v.embedding, dtype="float32") for v in vectors])

def search(q, k=3):
    qv = np.array(client.embeddings.create(model=E_MODEL, input=q).data[0].embedding, dtype="float32")
    sims = emb @ qv / (np.linalg.norm(emb,axis=1)*np.linalg.norm(qv)+1e-9)
    idx = np.argsort(-sims)[:k]
    return [docs[i] for i in idx]

def answer(question: str):
    passages = search(question, k=4)
    context = "\n\n".join(f"[{p['id']}] {p['text']}" for p in passages)
    prompt = f"Use the sources below to answer. If unsure, say so.\n\nSources:\n{context}\n\nQuestion: {question}"
    r = client.responses.create(model="gpt-4.1-mini", input=[{"role":"user","content":prompt}])
    print(r.output_text)

answer("Where is Furman, and what is CSC-475?")

```

Docs & concepts: embeddings for search, chunking, cosine similarity, and prompt-stuffing top results.  ￼


## 4) Multi-tool orchestrator: web + calc + RAG

Once you have multiple tools, keep a router loop: on each turn, let the model pick tools; you execute; loop until it emits a final message.

Common tools you’ll wire up:

•	Search/HTTP (fetch JSON/HTML and summarize),

•	Datastores (SQL query, vector search),

•	Side effects (send email, create calendar events) — be careful; require confirmation.

(Use the same function-tool pattern from §2 for each tool.)

## 5) Using the Agents SDK (optional but convenient)

OpenAI’s Agents SDK provides:

•	Lightweight agent loop with function tools as plain Python/TS functions (schema auto-generated),

•	Built-in tracing and state handling.

If you prefer batteries-included over writing the loop yourself, start here and register your tools; the SDK will handle function schema and orchestration.

## 6) Memory patterns

There’s no magical “long-term memory.” You implement it:

* Short-term: keep recent turns in the request.

* Episodic: store summaries of past sessions and attach them when the user returns.

* Profile/Prefs: keep key-value facts (e.g., “prefers verified quotes”), and prepend a small “profile block” to the system prompt.

* Vector memory: embed user notes / prior chats; retrieve by query each turn (same pattern as §3).

## 7) Streaming UX (recommended)
	•	Use the streaming API to render tokens as they arrive for snappy UX.
	•	If tools are used mid-stream, display a small status (“calling: calc(…) …done”), then continue streaming the model’s follow-up.
Streaming and event formats are in the API reference.  

## 8) Guardrails & safety
	•	Deterministic actions: For anything that spends money/sends data, require a confirmation step (“I’m about to email X—confirm?”).
	•	Schema validation: Keep strict JSON Schemas in your tools; reject invalid arguments and ask the model to retry.
	•	Allowlist outbound hosts for HTTP tools; sanitize inputs.
	•	Rate limiting & retries: Implement exponential backoff and idempotency keys on side-effecting calls.

## 9) Testing & eval (cheap but effective)
	•	Golden prompts: keep a list of questions + expected answers.
	•	Behavioral checks: does the agent pick the right tool? (Log tool calls; assert patterns.)
	•	Latency budget: record step timings (model, each tool).
	•	Hallucination guard: ask the model to provide citations when answering from RAG context; mark responses “uncited” if it can’t.

## 10) Deploy patterns
	•	Server: a thin HTTP/WS server that:
	•	validates the user,
	•	forwards messages to the agent loop,
	•	streams results back,
	•	stores transcripts & tool logs.
	•	Background jobs for long tool tasks (ETL, big web crawls); return a ticket and stream status updates.
	•	Observability: log every tool call, arguments, duration, and a sample of model inputs/outputs (with PII hygiene).

## End to End example

```python

# pip install openai numpy
import os, json, math, numpy as np
from openai import OpenAI
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# --- Tools ---
def calc(expression: str) -> str:
    return str(eval(expression, {"__builtins__": {}}, {"pi": math.pi, "e": math.e}))

CORPUS = [
  "Furman University is in Greenville, SC.",
  "CSC-475 is an applied data science capstone with BMW collaboration.",
  "Embeddings enable semantic search and RAG."
]
E_MODEL = "text-embedding-3-small"
EMB = np.vstack([client.embeddings.create(model=E_MODEL, input=CORPUS).data[i].embedding for i in range(len(CORPUS))])

def rag_search(query: str, k=3) -> str:
    qv = np.array(client.embeddings.create(model=E_MODEL, input=query).data[0].embedding)
    sims = EMB @ qv / (np.linalg.norm(EMB,axis=1)*np.linalg.norm(qv)+1e-9)
    idx = np.argsort(-sims)[:k]
    return "\n".join([f"- {CORPUS[i]}" for i in idx])

TOOLS = [
  {"type":"function","function":{
    "name":"calc","description":"Evaluate a math expression.",
    "parameters":{"type":"object","properties":{"expression":{"type":"string"}}, "required":["expression"]}
  }},
  {"type":"function","function":{
    "name":"rag_search","description":"Retrieve helpful facts for a user question.",
    "parameters":{"type":"object","properties":{"query":{"type":"string"}}, "required":["query"]}
  }}
]

def step(user_text: str):
    r = client.responses.create(model="gpt-4.1", tools=TOOLS, input=[{"role":"user","content":user_text}])
    while True:
        tool_call = next((b for b in r.output if b.type=="tool_call"), None)
        if not tool_call:
            print("".join(b.text for b in r.output if b.type=="output_text")); break
        name = tool_call.tool_name; args = json.loads(tool_call.arguments)
        result = calc(**args) if name=="calc" else rag_search(**args)
        r = client.responses.create(model="gpt-4.1",
            tools=TOOLS,
            input=[{"role":"user","content":user_text},
                   {"role":"tool","tool_call_id":tool_call.id,"content":result}]
        )

if __name__=="__main__":
    step("Using RAG, tell me what CSC-475 is.")
    step("Compute (sin(pi/2)**2 + cos(0)**2). Use the calculator.")

```